# 3 文本生成体验

本章从冒烟走向真实生成：构造一条小说续写请求（128 个输出 token），体验请求清单（request manifest）的完整格式、采样参数的作用，并观察 `enable_thinking` 关闭时的输出风格。


## 1 环境与工作区准备

复用第 2 章的统一环境与目录定位：推理仓克隆（cann-recipes-infer 固定基线 + 单卡 patch）与依赖环境均沿用第 2 章的准备结果，本章不再安装依赖。


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

print('[环境准备] 开始', flush=True)
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')

def locate_repo_root():
    repo_roots = []
    for env_name in ('GITCODE_REPO_ROOT', 'CANN_LEARNING_HUB_ROOT', 'REPO_ROOT'):
        if os.environ.get(env_name):
            repo_roots.append(Path(os.environ[env_name]))
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/longcat_next/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root. Open this notebook inside cann-learning-hub or set CANN_LEARNING_HUB_ROOT to the repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'longcat_next'
SRC_DIR = TUTORIAL_DIR / 'src'
SOURCE_DIR = REPO_ROOT / 'Sources' / 'model_inference_optimization' / 'longcat_next'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment before running this notebook.')
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' not in line:
        continue
    key, value = line.split('=', 1)
    if key == 'PATH':
        # login shell 会触发 conda init 并激活 base，其 PATH 会让后续 %%bash 单元里的
        # python3 解析到 base 环境（本教程环境没有 diffusers 等依赖）。
        # 跳过 PATH，保住当前 kernel 解释器的优先级。
        continue
    os.environ.__setitem__(key, value)
# 把 CANN bin 追加到当前 kernel 的 PATH（不覆盖），infer.sh 内的 python3 仍解析到本环境
_kernel_bin = str(Path(sys.executable).parent)
_cann_bin = str(Path(os.environ.get('ASCEND_HOME_PATH', '/usr/local/Ascend/ascend-toolkit')) / 'bin')
os.environ['PATH'] = f'{_kernel_bin}:{_cann_bin}:{os.environ["PATH"]}'

from inference_scripts.environment_check import find_unmet_requirements, read_pinned_requirements
from inference_scripts.recipe_workflow import (
    ensure_infer_repo,
    ensure_request_assets,
    prepare_runtime_yaml,
    print_yaml_focus,
    resolve_model_path,
)

# 复用（必要时自愈）第 2 章准备的克隆：固定基线 969160e + 单卡推理 patch
RECIPE_ROOT = ensure_infer_repo()
RECIPE_CONFIG_DIR = RECIPE_ROOT / 'models' / 'longcat_next' / 'config'
RECIPE_REQUESTS_DIR = RECIPE_ROOT / 'models' / 'longcat_next' / 'requests'
INFER_SH = RECIPE_ROOT / 'executor' / 'scripts' / 'infer.sh'
MODEL_PATH = resolve_model_path(quiet=True)
ensure_request_assets(MODEL_PATH, RECIPE_REQUESTS_DIR)

os.environ['LONGCAT_RECIPE_ROOT'] = str(RECIPE_ROOT)
os.environ['LONGCAT_MODEL_PATH'] = MODEL_PATH

print('infer 仓目录 =', RECIPE_ROOT)
print('模型权重 =', MODEL_PATH)
print('可见 NPU =', os.environ.get('ASCEND_RT_VISIBLE_DEVICES'))
print('[环境准备] 完成（依赖检查沿用第 2 章统一环境）', flush=True)


## 2 请求清单格式

LongCat-Next 样例的推理输入是一个 JSON **请求清单**（默认放在 `models/longcat_next/requests/` 下，YAML 的 `custom_params.multimodal.request_file` 指向它）。清单内请求**按顺序串行执行**，每条请求的字段如下：

| 字段 | 必填 | 说明 |
| --- | --- | --- |
| `id` | 是 | 请求标识，限 `[A-Za-z0-9_-]`、不超过 80 字符、清单内唯一；同时是结果子目录名 |
| `task` | 是 | 任务类型：`text` / `image_understanding` / `image_generation` / `audio_to_text` / `audio_to_audio` / `speech_synthesis` |
| `prompt` | 视任务 | 文本类任务必填 |
| `system` | 否 | 系统提示词；含 `reference_audio` 的任务由代码按官方模板填充 |
| `enable_thinking` | 否 | 聊天模板 thinking 开关（请求顶层，不能写进 `generation`），仅文本输出任务生效 |
| `media` | 视任务 | 素材路径字典，键名按任务固定，相对清单所在目录解析 |
| `generation` | 否 | 覆盖 YAML 中的默认采样参数 |
| `seed` | 否 | 随机种子，缺省取 `data_config.seed` |

`generation` 内可用参数：`max_new_tokens`、`min_new_tokens`、`do_sample`、`temperature`、`top_k`、`top_p`、`repetition_penalty`。注意**单条请求的 `max_new_tokens` 不得超过 YAML 的 `scheduler_config.max_new_tokens`**（该字段同时是 KV 预留容量的一部分）。

本章使用的 `requests/text.json`：一条小说续写请求，`do_sample=true`、`max_new_tokens=128`。


In [ ]:
from inference_scripts.recipe_workflow import read_request_manifest

TEXT_YAML_TEMPLATE = RECIPE_CONFIG_DIR / 'longcat_next_text_single_npu_offload.yaml'
text_requests = read_request_manifest(TEXT_YAML_TEMPLATE, recipe_root=RECIPE_ROOT)
print(json.dumps(text_requests, ensure_ascii=False, indent=2))


## 3 执行文本生成

准备运行 YAML 并启动推理。与冒烟相比，本例把 `scheduler_config.max_new_tokens` 放开到 128、`input_truncated_len` 扩到 256：KV 容量按 `max(input_truncated_len, max_prefill_tokens) + max_new_tokens` 预留，代码会自动对齐。128 个 token 在 CPU offload 模式下约需 5 分钟（每 token 约 2.3s，瓶颈为 CPU offload 的专家计算与搬运）。


In [ ]:
import os
import time
from inference_scripts.recipe_workflow import (
    collect_request_metrics,
    find_recipe_case_dir,
    prepare_runtime_yaml,
    print_yaml_focus,
)

TEXT_WORK_DIR = SOURCE_DIR / 'run_outputs' / 'text_generation'
# 运行 YAML 写入克隆仓 config/ 目录：上游 infer.sh 的 --yaml 只接受 config/ 下的文件名
TEXT_YAML = RECIPE_CONFIG_DIR / 'text_generation.yaml'
TEXT_WORK_DIR.mkdir(parents=True, exist_ok=True)

print('[文本生成] 开始', flush=True)
prepare_runtime_yaml(TEXT_YAML_TEMPLATE, TEXT_YAML, model_path=MODEL_PATH)
print_yaml_focus(TEXT_YAML)

os.environ['LONGCAT_RUN_YAML'] = str(TEXT_YAML)
os.environ['LONGCAT_RUN_YAML_NAME'] = TEXT_YAML.name  # infer.sh --yaml 传文件名
os.environ['LONGCAT_RUN_LOG'] = str(TEXT_WORK_DIR / 'infer_stdout.log')
os.environ['LONGCAT_RUN_START_TIME'] = str(time.perf_counter())
print('运行 YAML =', TEXT_YAML)


In [ ]:
%%bash
set -euo pipefail

echo 'recipes 启动命令:'
echo "cd ${LONGCAT_RECIPE_ROOT}"
echo "bash executor/scripts/infer.sh --model longcat_next --yaml ${LONGCAT_RUN_YAML_NAME}"
cd "${LONGCAT_RECIPE_ROOT}"
bash executor/scripts/infer.sh --model longcat_next --yaml "${LONGCAT_RUN_YAML_NAME}" 2>&1 | tee "${LONGCAT_RUN_LOG}"


## 4 查看生成结果与指标

读取 `result.json` 指标并查看小说续写文本；同时计算平均每 token 延迟（TPOT）。该数值可作为后续实验的文本生成基线。


In [ ]:
text_metrics = collect_request_metrics(
    RECIPE_ROOT,
    TEXT_WORK_DIR,
    'text_generation',
    TEXT_YAML,
)
for row in text_metrics['requests']:
    tpot = row['autoregressive_seconds'] / max(row['generated_sequence_tokens'], 1)
    print(f"{row['id']}: {row['generated_sequence_tokens']} tok, "
          f"{row['autoregressive_seconds']:.1f} s, TPOT≈{tpot:.2f} s/tok")

story = (Path(text_metrics['recipe_case_dir']) / 'results' / 'text_story' / 'text.txt').read_text(encoding='utf-8')
print('\n生成文本:')
print(story)


## 5 采样参数说明

本例使用的采样参数及其效果：

| 参数 | 本例取值 | 说明 |
| --- | --- | --- |
| `do_sample` | `true` | 开启采样；`false` 为贪心解码，输出确定 |
| `temperature` | `0.7` | 越低越确定，越高越发散 |
| `top_p` | `0.9` | 核采样，只在累计概率前 90% 的词中采样 |
| `repetition_penalty` | `1.05` | 轻度重复惩罚，抑制长文本复读 |

可以尝试修改 `requests/text.json` 中的 `temperature`（如 0.2 / 1.2）或把 `do_sample` 设为 `false`，重新运行本章观察输出差异。注意所有输出都会受到 `max_new_tokens=128` 截断。

> **提示**：实验表明，`enable_thinking=false` 时**图像理解任务**若使用开放式中文长 prompt（如"请详细描述这张图片的内容。"）可能直接输出结束符（1 个空 token）。建议图像理解使用简短问句（如官方示例 "What book is this?"），或显式设置 `enable_thinking: true`；第 4 章的请求清单已按此约束编写。
